In [ ]:
# %uv pip install pandas numpy matplotlib seaborn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. データの読み込み

In [ ]:
df_raw = pd.read_excel("data/data.xlsx")

In [ ]:
df_raw.head(3)

In [ ]:
df_raw.shape

In [ ]:
df_raw.dtypes

In [ ]:
list_numeric_cols = df_raw.select_dtypes(include="number").columns.tolist()
list_non_numeric_cols = df_raw.select_dtypes(exclude="number").columns.tolist()

In [ ]:
list_numeric_cols

In [ ]:
list_non_numeric_cols

## 2. データの中身の確認

In [ ]:
df_eda = df_raw.copy()

## 2.1 データの概要の確認

In [ ]:
# nullの確認
df_eda.isnull().sum()

In [ ]:
# 数値データの概要確認
df_eda[list_numeric_cols].describe()

# Countは、mean:1.0, std:0.0, min:1.0, 25%:1.0, 50%:1.0, 75%:1.0, max:1.01.00.であり、全部1.0だと推測される。情報量がないので分析では使わない。

In [ ]:
# 数値以外のデータの概要確認
df_eda[list_non_numeric_cols].describe()

### データ概要の確認メモ
* CustomerID→ 全サンプルでunique。ただし、strになっている。 →数値＋"-"+アルファベットで構成されていた 
* Country, State→全サンプルで、United States, California。情報量がないので、分析の時には削除  
* Total Charges→金額のはずが、dataframeでは数値以外の扱いになっている。中身確認。  
* Lat Long→全7043件のうち、uniqueが1652。一致しているサンプルが一定数ある。CustomerIDは全サンプルでuniqueのはずなので、Lat Longが一致している理由が不明。分析の時には一旦除外する？Lat Longから地域を判定して、上流、中流、など区画レベルのクラスタリングの間接的な指標になる可能性はあるが、今回はわかりやすい他の特徴量を優先。→zipcodeで大まかに区画はわかるから、lat longは分析では除外


### 2.2 データを個別に確認

In [ ]:
# チャーンしたかどうかを示す目的変数、Churn Labelを確認
# なお、Churn Valueとは強い関係性がある
df_eda["Churn Label"].value_counts()

# メモ：
# 不均衡データ↓
# No     5174
# Yes    1869

In [ ]:
# CustomerID
df_eda["CustomerID"].head(5)

In [ ]:
# Zip Code
df_eda["Zip Code"].value_counts()

In [ ]:
df_eda["isnum_Total Charges"] = pd.to_numeric(df_eda["Total Charges"], errors="coerce").notna()

In [ ]:
len(df_eda[["CustomerID","Total Charges"]][df_eda["isnum_Total Charges"]==False])

# Total Chargesの中身が空白が11件ある→少量なので、分析の時には、補完せずに削除する

## 3. 目的変数とカテゴリ変数のクロス集計

In [ ]:
y_col = "Churn Label"

### 3.1 City

In [ ]:
pd.crosstab(
    df_eda["City"],
    df_eda[y_col],
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)


# そもそもcityの母数が少ないcityもあるので、件数も確認する
# pd.crosstab(
#     df_eda["City"],
#     df_eda[y_col],
#     margins=True,
# ).sort_values(by="Yes", ascending=False)

### 3.2 Senior Citizen

In [ ]:
# 65歳以上かどうかとチャーン確認
pd.crosstab(df_eda["Senior Citizen"], df_eda[y_col], normalize=True).sort_values(by="Yes", ascending=False)

### 3.3 Dependents

In [ ]:
# 家族構成の有無とチャーン確認
pd.crosstab(df_eda["Dependents"], df_eda[y_col], normalize=True).sort_values(by="Yes", ascending=False)

### 3.4 契約回線の確認

In [ ]:
# 電話の契約有無とチャーン確認

pd.crosstab(
    df_eda["Phone Service"], 
    df_eda[y_col], 
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)


In [ ]:
# Internet Serviceの契約有無とチャーン確認

# Internet契約の有無とチャーンの関係を確認したいので、Fiber opticとDSLをまとめてYesにする
df_tmp = df_eda.copy()
df_tmp["Internet Service"] = df_tmp["Internet Service"].replace({"Fiber optic": "Yes", "DSL":"Yes"})

pd.crosstab(
    df_tmp["Internet Service"], 
    df_tmp[y_col], 
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)


In [ ]:
# Phone ServiceとInternet Serviceの契約有無で確認

pd.crosstab(
    index = [df_tmp["Phone Service"], df_tmp["Internet Service"]], 
    columns = df_tmp[y_col], 
    margins=True,
    normalize = "index"
).sort_values(by="Yes", ascending=False)

### 3.5 回線以外の契約サービスの確認

In [ ]:
# 回線以外の付随サービスの契約がチャーンに影響があるか確認

In [ ]:
pd.crosstab(
    df_eda["Online Security"], 
    df_eda[y_col], 
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)

# メモ
# Online Securityなし → 約41.7%がチャーン
# Online Securityあり → 約14.6%がチャーン
# インターネット契約なし → 約7.4%がチャーン


In [ ]:
pd.crosstab(
    df_eda["Online Backup"], 
    df_eda[y_col], 
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)

# メモ
# Online Backupなし→約39.9がチャーン
# Online Backupあり→約21.5%がチャーン
# インターネット契約なし → 約7.4%がチャーン


In [ ]:
pd.crosstab(
    df_eda["Device Protection"], 
    df_eda[y_col], 
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)

# メモ
# Device Protectionなし→約39.1%がチャーン
# Device Protectionあり→約22.5%がチャーン



In [ ]:
pd.crosstab(
    df_eda["Tech Support"], 
    df_eda[y_col], 
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)

# メモ
# Tech Supportなし→約41.6%がチャーン
# Tech Supportあり→約15.1%がチャーン


In [ ]:
# 回線以外の付随サービスの契約有無は、チャーンへの影響が大きい可能性大

### 3.5 回線の使用用途とチャーンの確認

In [ ]:
pd.crosstab(
    df_eda["Streaming TV"], 
    df_eda[y_col], 
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)

# メモ：
# TVの使用用途ではチャーンに大きな影響はない


In [ ]:
pd.crosstab(
    df_eda["Streaming Movies"],
    df_eda[y_col],
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)

# メモ：
# Movieの使用用途ではチャーンに大きな影響はない

In [ ]:
# 回線の使用用途はチャーンへの影響は無い可能性が高い

### 3.6 Contract

In [ ]:
pd.crosstab(
    df_eda["Contract"],
    df_eda[y_col],
    margins=True,
    normalize="index"
).sort_values(by="Yes", ascending=False)

## 4. 目的変数と数値変数の確認

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=df_eda, 
    x="Tenure Months", hue=y_col, 
    multiple="stack",  #　layerだと重なって別の色に見えたので、stackにする
    alpha=0.8, 
    bins="fd"
)

plt.ylabel("Frequency")
plt.show()


In [ ]:
# Total Charges
# 数値だけの行を取り出す

df_tc = df_eda[[y_col, "Total Charges"]][df_eda["isnum_Total Charges"]==True]
df_tc["Total Charges"] = df_tc["Total Charges"].astype(int) # int型にして小数点以下をまるめる

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=df_tc, 
    x="Total Charges", hue=y_col, 
    multiple="stack", 
    alpha=0.8, 
    bins="fd"
)

plt.ylabel("Frequency")
plt.show()

## 5. データの保存

In [ ]:
# df_edaで前処理したデータを保存
# modelでfeature_importanceやSHAPを見る際に一手間省ける

save_cols = ["CustomerID", "Zip Code", "Tenure Months", "Monthly Charges", 
             "City", "Gender", "Senior Citizen", "Partner", "Dependents", "Phone Service", "Multiple Lines",
             "Internet Service", "Online Security", "Online Backup", "Device Protection", "Tech Support", 
             "Streaming TV", "Streaming Movies", "Contract", "Paperless Billing", "Payment Method", "Total Charges", 
             "Churn Label","isnum_Total Charges"
             ]

df_save = df_eda[save_cols].copy()

df_save = df_save[df_save["isnum_Total Charges"]==True]
df_save["Total Charges"] = df_save["Total Charges"].astype(float) 

df_save.to_csv("./data/prepro_data.csv", index=False, encoding="utf-8")

In [ ]:
df_save.head(3)